# AlertaBairro: especialização do modelo com LoRA

**Disciplina:** IA Generativa, Global Solution FIAP 2026.2 (CitizenTech)

**Objetivo.** Ajustar o `Qwen2.5-3B-Instruct`, o mesmo modelo que o site usa no Ollama, para seguir o **contrato do assistente**:
- responder só com os trechos recebidos, com o número exato;
- copiar em `fontes` o id do trecho;
- recusar quando os trechos não respondem;
- devolver sempre JSON válido.

**Por que LoRA.** Na avaliação da RAG simples (`avaliacao/resultados/placar_rag.jsonl`), o modelo base entendeu o conteúdo, mas errou o formato:
- inventou ids como `sub:mes2026-05` e `doc:limites_dados#5`;
- 14% das respostas foram barradas pela validação.

O LoRA treina só adaptadores pequenos, então cabe numa GPU T4 gratuita do Colab, e o resultado pode ser exportado em GGUF para rodar no mesmo Ollama.

**Dados.** `treino/dados_treino.jsonl` e `treino/dados_validacao.jsonl` são gerados por `treino/gerar_dados_treino.py`:
- usam o mesmo prompt e a mesma busca BM25 do site;
- não contêm nenhuma das 50 perguntas de teste (o script mede a similaridade e descarta as parecidas).

**Como rodar.** Menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois, rode as células em ordem.

## 1. Conferir a GPU

In [1]:
!nvidia-smi

Tue Nov 17 23:50:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Instalar o Unsloth
Biblioteca de ajuste fino eficiente (LoRA com o modelo em 4 bits). Leva alguns minutos.

In [2]:
!pip install -q unsloth

## 3. Baixar o repositório (dados de treino e código de validação do site)

In [3]:
!rm -rf AlertaBairro
!git clone -q -b genai-avaliacao --depth 1 https://github.com/GeovannaSCunha/AlertaBairro.git
!wc -l AlertaBairro/treino/dados_treino.jsonl AlertaBairro/treino/dados_validacao.jsonl
!cat AlertaBairro/treino/estatisticas.json | head -20

   212 AlertaBairro/treino/dados_treino.jsonl
    23 AlertaBairro/treino/dados_validacao.jsonl
   235 total
{
  "semente": 42,
  "total": 235,
  "treino": 212,
  "validacao": 23,
  "respostas": 190,
  "recusas": 45,
  "por_grupo": {
    "sub": 90,
    "indice": 28,
    "mes": 26,
    "chuva": 20,
    "kpi": 3,
    "clima": 3,
    "doc": 20,
    "recusa": 45
  },
  "limite_similaridade_com_teste": 0.6,
  "maior_similaridade_mantida": 0.6,
  "descartadas_por_parecer_teste": [


## 4. Carregar o modelo base em 4 bits

In [4]:
import sys, json, time, torch
sys.path.insert(0, "AlertaBairro/backend")
from unsloth import FastLanguageModel

SEMENTE = 42
MAX_TOKENS = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_TOKENS,
    load_in_4bit=True,
)
print(model.config._name_or_path, "| GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1117 23:55:14.397000 12950 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1117 23:55:14.437000 12950 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.6: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit | GPU: Tesla T4


## 5. Função de avaliação no conjunto de validação

Mede, nos exemplos de validação (que não entram no treino):
- **JSON válido:** usa o mesmo validador do site, `app.assistente.llm.validar_saida`, que barra JSON quebrado e ids fora do contexto;
- **fontes exatas:** o modelo citou exatamente os ids esperados;
- **recusa correta:** o campo `recusou` bate com o esperado.

In [5]:
from app.assistente.llm import validar_saida, ErroLLM

def carregar(caminho):
    return [json.loads(l) for l in open(caminho, encoding="utf-8") if l.strip()]

validacao = carregar("AlertaBairro/treino/dados_validacao.jsonl")

def ids_do_contexto(texto_usuario):
    return {l[1:l.index("]")] for l in texto_usuario.splitlines() if l.startswith("[")}

def avaliar(model, exemplos, rotulo):
    FastLanguageModel.for_inference(model)
    json_ok = fontes_ok = recusa_ok = 0
    amostras = []
    for ex in exemplos:
        msgs = ex["messages"][:-1]
        esperado = json.loads(ex["messages"][-1]["content"])
        entrada = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            saida = model.generate(input_ids=entrada, max_new_tokens=300, do_sample=False)
        texto = tokenizer.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True)
        try:
            obj = validar_saida(texto, ids_do_contexto(msgs[1]["content"]))
            json_ok += 1
            fontes_ok += sorted(obj["fontes"]) == sorted(esperado["fontes"])
            recusa_ok += obj["recusou"] == esperado["recusou"]
        except ErroLLM as exc:
            texto = f"[barrado: {exc}] {texto}"
        amostras.append(texto)
    n = len(exemplos)
    print(f"{rotulo}: JSON válido {json_ok}/{n} ({json_ok/n:.0%}) | fontes exatas {fontes_ok}/{n} ({fontes_ok/n:.0%}) "
          f"| recusa correta {recusa_ok}/{n} ({recusa_ok/n:.0%})")
    for t in amostras[:3]:
        print("  ->", t[:200])
    return {"json_valido": json_ok / n, "fontes_exatas": fontes_ok / n, "recusa_correta": recusa_ok / n}

## 6. Avaliar o modelo base, antes do treino

In [6]:
antes = avaliar(model, validacao, "ANTES (modelo base)")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

ANTES (modelo base): JSON válido 20/23 (87%) | fontes exatas 10/23 (43%) | recusa correta 20/23 (87%)
  -> {"resposta": "Casa Verde/Cachoeirinha registrou 4 alagamentos na base histórica.", "fontes": ["sub:CASAVERDECACHOEIRINHA"], "recusou": false}
  -> [barrado: Citações fora do contexto: ['sub:SAOMIGUELPAULISTA']] {
  "resposta": "Em setembro de 2026, a base histórica registra 95 ocorrências de alagamento.",
  "fontes": ["sub:SAOMIGUELPAULISTA"],
 
  -> {"resposta": "M'Boi Mirim ocupa a posição 4 no ranking de alagamentos entre as subprefeituras de São Paulo.", "fontes": ["sub:MBOIMIRIM", "indice:MBOIMIRIM"], "recusou": false}


## 7. Adicionar os adaptadores LoRA
Só cerca de 1% dos parâmetros são treinados. O modelo base fica congelado.

In [7]:
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEMENTE,
)
treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {treinaveis:,} de {total:,} ({treinaveis/total:.2%})")

Unsloth 2026.11.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Parâmetros treináveis: 29,933,568 de 1,830,055,936 (1.64%)


## 8. Preparar os dados no formato de conversa do Qwen

In [8]:
from datasets import Dataset

def para_texto(exemplos):
    return [{"text": tokenizer.apply_chat_template(e["messages"], tokenize=False)} for e in exemplos]

treino = carregar("AlertaBairro/treino/dados_treino.jsonl")
ds_treino = Dataset.from_list(para_texto(treino))
ds_validacao = Dataset.from_list(para_texto(validacao))
tamanhos = [len(tokenizer(t["text"]).input_ids) for t in ds_treino]
print(f"treino: {len(ds_treino)} exemplos | validação: {len(ds_validacao)} | tokens por exemplo: média {sum(tamanhos)/len(tamanhos):.0f}, máx {max(tamanhos)}")

treino: 212 exemplos | validação: 23 | tokens por exemplo: média 1058, máx 1322


## 9. Treinar

A perda é calculada **só na resposta do assistente**, com `train_on_responses_only`: o modelo aprende a responder, e não a repetir o prompt.

In [9]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_treino,
    eval_dataset=ds_validacao,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_TOKENS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy="epoch",
        output_dir="saida_lora",
        seed=SEMENTE,
        report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
inicio = time.time()
estatisticas = trainer.train()
print(f"Tempo de treino: {(time.time()-inicio)/60:.1f} min | perda final de treino: {estatisticas.training_loss:.4f}")
for registro in trainer.state.log_history:
    if "eval_loss" in registro:
        print(f"época {registro['epoch']:.0f}: perda na validação = {registro['eval_loss']:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/212 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/212 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/23 [00:00<?, ? examples/s]

Unsloth: Packing is on by default - documents are packed into full rows with no cross-document attention. Set packing = False in SFTConfig (or UNSLOTH_DEFAULT_PACKING=0) to turn it off.


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

... (2 outputs left out)

Epoch,Training Loss,Validation Loss
1,0.261056,0.153977
2,0.021975,0.064732
3,0.017009,0.059751


Filter:   0%|          | 0/23 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in saida_lora/checkpoint-57/tokenizer_config.json.


Tempo de treino: 15.0 min | perda final de treino: 0.1442
época 1: perda na validação = 0.1540
época 2: perda na validação = 0.0647
época 3: perda na validação = 0.0598


## 10. Avaliar o modelo ajustado, depois do treino

In [10]:
depois = avaliar(model, validacao, "DEPOIS (LoRA)")
print()
print(f"{'métrica':<16}{'antes':>8}{'depois':>8}")
for k in antes:
    print(f"{k:<16}{antes[k]:>8.0%}{depois[k]:>8.0%}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

DEPOIS (LoRA): JSON válido 23/23 (100%) | fontes exatas 22/23 (96%) | recusa correta 23/23 (100%)
  -> {"resposta": "Segundo a base histórica, Casa Verde/Cachoeirinha tem 4 registros de alagamento.", "fontes": ["sub:CASAVERDECACHOEIRINHA"], "recusou": false}
  -> {"resposta": "O total de setembro de 2026 na base histórica é 95 registros de alagamento.", "fontes": ["mes:2026-09"], "recusou": false}
  -> {"resposta": "M'Boi Mirim ocupa a posição 4 entre 30 subprefeituras com registros. O ranking mostra a distribuição dos registros, não um índice de risco.", "fontes": ["sub:MBOIMIRIM"], "recusou": fals

métrica            antes  depois
json_valido          87%    100%
fontes_exatas        43%     96%
recusa_correta       87%    100%


## 11. Exportar para o Ollama (GGUF q4_k_m) e salvar no Google Drive

O modelo é exportado com os adaptadores já fundidos, na mesma quantização (`q4_k_m`) do `qwen2.5:3b-instruct` que roda no Ollama. Assim a comparação é justa. O adaptador LoRA sozinho (poucos MB) também é salvo.

In [11]:
model.save_pretrained("alertabairro_lora_adaptador")
tokenizer.save_pretrained("alertabairro_lora_adaptador")
model.save_pretrained_gguf("alertabairro_lora_gguf", tokenizer, quantization_method="q4_k_m")
!ls -la alertabairro_lora_adaptador alertabairro_lora_gguf

Unsloth: Restored added_tokens_decoder metadata in alertabairro_lora_adaptador/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in alertabairro_lora_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


... (9 outputs left out)



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:23<00:00, 72.00s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:41<00:00, 50.66s/it]


Unsloth: Merge process complete. Saved to `/content/alertabairro_lora_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12347-mix-241a0f3 (app-b12347-mix-241a0f3-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['alertabairro_lora_gguf_gguf/Qwen2.5-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversi

In [12]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/AlertaBairro_LoRA
!cp -v alertabairro_lora_gguf/*.gguf /content/drive/MyDrive/AlertaBairro_LoRA/
!cp -rv alertabairro_lora_adaptador /content/drive/MyDrive/AlertaBairro_LoRA/
json.dump({"antes": antes, "depois": depois}, open("/content/drive/MyDrive/AlertaBairro_LoRA/metricas_validacao.json", "w"), indent=2)
!ls -la /content/drive/MyDrive/AlertaBairro_LoRA

Mounted at /content/drive
cp: cannot stat 'alertabairro_lora_gguf/*.gguf': No such file or directory
'alertabairro_lora_adaptador' -> '/content/drive/MyDrive/AlertaBairro_LoRA/alertabairro_lora_adaptador'
'alertabairro_lora_adaptador/README.md' -> '/content/drive/MyDrive/AlertaBairro_LoRA/alertabairro_lora_adaptador/README.md'
'alertabairro_lora_adaptador/adapter_model.safetensors' -> '/content/drive/MyDrive/AlertaBairro_LoRA/alertabairro_lora_adaptador/adapter_model.safetensors'
'alertabairro_lora_adaptador/adapter_config.json' -> '/content/drive/MyDrive/AlertaBairro_LoRA/alertabairro_lora_adaptador/adapter_config.json'
'alertabairro_lora_adaptador/chat_template.jinja' -> '/content/drive/MyDrive/AlertaBairro_LoRA/alertabairro_lora_adaptador/chat_template.jinja'
'alertabairro_lora_adaptador/tokenizer_config.json' -> '/content/drive/MyDrive/AlertaBairro_LoRA/alertabairro_lora_adaptador/tokenizer_config.json'
'alertabairro_lora_adaptador/tokenizer.json' -> '/content/drive/MyDrive/AlertaB

In [13]:
!cp -v alertabairro_lora_gguf_gguf/*.gguf alertabairro_lora_gguf_gguf/Modelfile /content/drive/MyDrive/AlertaBairro_LoRA/
!ls -la /content/drive/MyDrive/AlertaBairro_LoRA

'alertabairro_lora_gguf_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf' -> '/content/drive/MyDrive/AlertaBairro_LoRA/Qwen2.5-3B-Instruct.Q4_K_M.gguf'
'alertabairro_lora_gguf_gguf/Modelfile' -> '/content/drive/MyDrive/AlertaBairro_LoRA/Modelfile'
total 1884686
drwx------ 3 root root       4096 Nov 18 00:35 .
drwx------ 7 root root       4096 Nov 18 00:30 ..
drwx------ 2 root root       4096 Nov 18 00:30 alertabairro_lora_adaptador
-rw------- 1 root root        252 Nov 18 00:30 metricas_validacao.json
-rw------- 1 root root       1722 Nov 18 00:35 Modelfile
-rw------- 1 root root 1929902688 Nov 18 00:35 Qwen2.5-3B-Instruct.Q4_K_M.gguf
